<a href="https://colab.research.google.com/github/MeronEDS/urban_heat_island_change/blob/main/scripts/1_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

cleaning data scripts

In [1]:
from google.colab import drive

startingn to do data cleaning

change runtime back to R

In [51]:
setwd ("/content/drive/MyDrive/urban_heat_island_change")
getwd()


[1] "/content/drive/MyDrive/urban_heat_island_change"

In [57]:
#load library
library(readr)
library(dplyr)
library(tidyr)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ purrr     1.2.2
✔ ggplot2   4.0.3     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [58]:
# read in raw data

surface_temp_raw <- read_csv("raw_data/temperature_raw.csv")
station_raw <- read_csv("raw_data/weather_stations_raw.csv")
list.files()

Rows: 2252 Columns: 15
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): station_id, measurement
dbl (13): year, Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 13 Columns: 263
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (7): station_id, station_name, location, station_setting, subcity_name...
dbl (256): elevation_m, subcity_area_km2, initiative_start_year, distance_to...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] "clean_data" "figures"    "raw_data"   "results"    "scripts"

In [59]:
head(surface_temp_raw)

station_id,year,measurement,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ST01,2005,max_air_temp_c,23.31,24.54,24.92,NA,24.35,22.74,21.86,21.09,21.20,21.99,22.37,23.24
st01,2005,max_air_temp_c,NA,NA,NA,24.72,NA,NA,NA,NA,NA,NA,NA,NA
ST01,2005,min_air_temp_c,10.73,11.51,11.99,NA,11.13,14.71,13.87,12.74,12.64,8.97,9.50,10.03
st01,2005,min_air_temp_c,NA,NA,NA,11.82,NA,NA,NA,NA,NA,NA,NA,NA
ST01,2005,mean_air_temp_c,17.04,18.01,18.47,NA,17.74,18.70,17.86,16.89,16.89,15.46,15.95,16.63
st01,2005,mean_air_temp_c,NA,NA,NA,-99.00,NA,NA,NA,NA,NA,NA,NA,NA


In [55]:
head(station_raw)

station_id,station_name,location,elevation_m,station_setting,subcity_name,city_ring,subcity_area_km2,nearest_green_initiative,initiative_start_year,⋯,initiative_active_2016,initiative_active_2017,initiative_active_2018,initiative_active_2019,initiative_active_2020,initiative_active_2021,initiative_active_2022,initiative_active_2023,initiative_active_2024,initiative_active_2025
<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ST01,Bole Airport,"8.981, 38.799",2354,Urban,Bole,Middle,80.0,Riverside (Beautifying Sheger),2019,⋯,0,0,0,1,1,1,1,1,1,1
ST02,Arat Kilo,"9.034, 38.763",2410,urban,Arada,Inner,9.9,Unity Park,2019,⋯,0,0,0,1,1,1,1,1,1,1
ST03,Entoto,"9.083, 38.760",2790,Peri-urban,Gullele,Middle,30.2,Entoto Park,2020,⋯,0,0,0,0,1,1,1,1,1,1
ST04,Kotebe,"9.030, 38.862",2380,Urban,Yeka,Middle,68.0,Corridor Development Phase 1,2024,⋯,0,0,0,0,0,0,0,0,1,1
ST05,Akaki,"8.872, 38.789",2080,Peri urban,Akaky Kaliti,Outer,118.1,Corridor Development Phase 2,2024,⋯,0,0,0,0,0,0,0,0,1,1
ST06,Kolfe,"9.030, 38.700",2450,Urban,Kolfe Keranio,Middle,61.3,Corridor Development Phase 2,2024,⋯,0,0,0,0,0,0,0,0,1,1


In [64]:
# Pivoting the urban_heat_data to be tidy
 # TEMPERATURE: fix IDs
temp_clean <- surface_temp_raw |>
  mutate(station_id = str_to_upper(str_trim(station_id)),       # st01 -> ST01
         year = as.integer(year)) |>
  distinct() |>                                                  # remove duplicate rows
  pivot_longer(Jan:Dec, names_to = "month", values_to = "value",
               values_drop_na = TRUE) |>                         # months to rows, drop empty cells
  pivot_wider(names_from = measurement, values_from = value) |>  # one column per measurement
  mutate(month = match(month, month.abb))
head(temp_clean)

station_id,year,month,max_air_temp_c,min_air_temp_c,mean_air_temp_c,highest_max_air_temp_c,lowest_min_air_temp_c,days_with_data,land_surface_temp_c,clear_satellite_passes
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ST01,2005,1,23.31,10.73,17.04,24.7,7.1,30,34.70,2
ST01,2005,2,24.54,11.51,18.01,26.6,8.1,27,35.16,2
ST01,2005,3,24.92,11.99,18.47,27.5,10.0,28,33.86,2
ST01,2005,5,24.35,11.13,17.74,26.4,8.0,30,34.21,1
ST01,2005,6,22.74,14.71,18.70,26.2,12.5,29,NA,0
ST01,2005,7,21.86,13.87,17.86,23.8,11.4,28,NA,0


In [66]:

# cleaning the station data
weather_station_clean <- station_raw |>
  mutate(station_id = str_to_upper(str_trim(station_id)),
         station_setting = station_setting |> str_trim() |> str_to_lower() |>
           str_replace(" ", "-")) |>                             # -> urban / peri-urban / rural
  separate_wider_delim(location, delim = ", ",
                       names = c("latitude", "longitude")) |>
  mutate(across(c(latitude, longitude), as.numeric)) |>
  pivot_longer(matches("_\\d{4}$"),
               names_to = c(".value", "year"),
               names_pattern = "(.*)_(\\d{4})",
               names_transform = list(year = as.integer))
head(weather_station_clean)

station_id,station_name,latitude,longitude,elevation_m,station_setting,subcity_name,city_ring,subcity_area_km2,nearest_green_initiative,⋯,pct_cropland_1km,pct_built_up_1km,pct_bare_1km,pct_water_1km,mean_ndvi_1km,mean_ndbi_1km,population_1km,green_space_ha_1km,green_space_m2_per_person_1km,initiative_active
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ST01,Bole Airport,8.981,38.799,2354,urban,Bole,Middle,80,Riverside (Beautifying Sheger),⋯,10.2,55.2,10.0,1.1,0.213,0.030,46840,74.2,15.8,0
ST01,Bole Airport,8.981,38.799,2354,urban,Bole,Middle,80,Riverside (Beautifying Sheger),⋯,9.0,56.8,9.6,1.0,0.201,0.033,48250,74.1,15.4,0
ST01,Bole Airport,8.981,38.799,2354,urban,Bole,Middle,80,Riverside (Beautifying Sheger),⋯,8.5,57.8,9.4,1.0,0.190,0.008,49700,73.5,14.8,0
ST01,Bole Airport,8.981,38.799,2354,urban,Bole,Middle,80,Riverside (Beautifying Sheger),⋯,8.2,57.9,9.8,1.1,0.213,0.027,51190,72.3,14.1,0
ST01,Bole Airport,8.981,38.799,2354,urban,Bole,Middle,80,Riverside (Beautifying Sheger),⋯,7.5,58.9,9.2,1.0,0.209,0.003,52720,73.5,13.9,0
ST01,Bole Airport,8.981,38.799,2354,urban,Bole,Middle,80,Riverside (Beautifying Sheger),⋯,7.1,60.1,8.6,1.0,0.217,0.032,54310,72.8,13.4,0


In [69]:
# join data sets
uhi_clean <- left_join(temp_clean, weather_station_clean, by = c("station_id", "year"))
head(uhi_clean)

station_id,year,month,max_air_temp_c,min_air_temp_c,mean_air_temp_c,highest_max_air_temp_c,lowest_min_air_temp_c,days_with_data,land_surface_temp_c,⋯,pct_cropland_1km,pct_built_up_1km,pct_bare_1km,pct_water_1km,mean_ndvi_1km,mean_ndbi_1km,population_1km,green_space_ha_1km,green_space_m2_per_person_1km,initiative_active
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ST01,2005,1,23.31,10.73,17.04,24.7,7.1,30,34.70,⋯,10.2,55.2,10,1.1,0.213,0.03,46840,74.2,15.8,0
ST01,2005,2,24.54,11.51,18.01,26.6,8.1,27,35.16,⋯,10.2,55.2,10,1.1,0.213,0.03,46840,74.2,15.8,0
ST01,2005,3,24.92,11.99,18.47,27.5,10.0,28,33.86,⋯,10.2,55.2,10,1.1,0.213,0.03,46840,74.2,15.8,0
ST01,2005,5,24.35,11.13,17.74,26.4,8.0,30,34.21,⋯,10.2,55.2,10,1.1,0.213,0.03,46840,74.2,15.8,0
ST01,2005,6,22.74,14.71,18.70,26.2,12.5,29,NA,⋯,10.2,55.2,10,1.1,0.213,0.03,46840,74.2,15.8,0
ST01,2005,7,21.86,13.87,17.86,23.8,11.4,28,NA,⋯,10.2,55.2,10,1.1,0.213,0.03,46840,74.2,15.8,0


In [70]:
# keeping the cleaned data in clean_data folder
write_csv(uhi_clean,"clean_data/uhi_change_clean")